# 📊 Retail Intelligence Platform — 02: Baseline Models & Validation Benchmark

**Objetivo:** Establecer la partición temporal purgada sin *data leakage* y evaluar modelos de referencia heurísticos (*Naive*, *Seasonal Naive* y *Moving Average*) para definir la "vara mínima" que el modelo de Machine Learning deberá superar.

---

### 📋 Módulos Evaluados:
- `src/retail_platform/models/split.py` $ightarrow$ `TemporalSplitter`
- `src/retail_platform/evaluation/metrics.py` $ightarrow$ `RMSLE`, `MAE`, `RMSE`, `WAPE`, `Bias`
- `src/retail_platform/models/baseline.py` $ightarrow$ `NaiveBaseline`, `SeasonalNaiveBaseline`, `MovingAverageBaseline`

In [ ]:
# Imports y configuracion
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from retail_platform.models.split import TemporalSplitter
from retail_platform.models.baseline import (
    NaiveBaseline, 
    SeasonalNaiveBaseline, 
    MovingAverageBaseline, 
    evaluate_all_baselines
)
from retail_platform.evaluation.metrics import evaluate_predictions

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (14, 5)
plt.rcParams['font.size'] = 11

print("Modulos importados exitosamente.")

## 1. Carga de Datos y Partición Temporal Purgada

In [ ]:
# Carga del feature dataset generado en la Fase 3
data_path = Path('../data/processed/features_master.parquet')
df = pd.read_parquet(data_path)

print(f"Dataset cargado: {len(df):,} filas | {df.shape[1]} columnas")

# Aplicar division temporal estricta (16 dias de validacion)
splitter = TemporalSplitter(val_days=16)
train_df, val_df, test_df = splitter.split(df)

## 2. Benchmark de Modelos Baseline

In [ ]:
# Evaluacion consolidada de todos los baselines
baseline_benchmark = evaluate_all_baselines(val_df)
baseline_benchmark

## 3. Comparativa Visual de Rendimiento & Métricas de Negocio

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5))

# Grafico de RMSLE (Menor es mejor)
sns.barplot(data=baseline_benchmark, x='Modelo', y='RMSLE', palette='Blues_r', ax=ax1)
ax1.set_title('Comparativa de Error de Optimizacion (RMSLE)', fontsize=13, weight='bold')
ax1.set_ylabel('RMSLE (Menor es mejor)')
ax1.tick_params(axis='x', rotation=15)
for i, v in enumerate(baseline_benchmark['RMSLE']):
    ax1.text(i, v + 0.01, f'{v:.4f}', ha='center', fontweight='bold')

# Grafico de WAPE (%) (Error porcentual ponderado sobre volumen real)
sns.barplot(data=baseline_benchmark, x='Modelo', y='WAPE (%)', palette='Reds_r', ax=ax2)
ax2.set_title('Error Porcentual de Volumen Retail (WAPE %)', fontsize=13, weight='bold')
ax2.set_ylabel('WAPE % (Menor es mejor)')
ax2.tick_params(axis='x', rotation=15)
for i, v in enumerate(baseline_benchmark['WAPE (%)']):
    ax2.text(i, v + 0.5, f'{v:.2f}%', ha='center', fontweight='bold')

plt.tight_layout()
plt.show()

## 4. Inspección Visual: Ventas Reales vs. Predicción Baseline

Visualizamos el pronóstico sobre la familia de mayor venta (`GROCERY I`) en la Tienda 1 durante los 16 días de validación.

In [ ]:
# Filtrar serie de ejemplo: Tienda 1, GROCERY I
sample_val = val_df[(val_df['store_nbr'] == 1) & (val_df['family'] == 'GROCERY I')].sort_values('date')

s_naive = SeasonalNaiveBaseline()
naive = NaiveBaseline()

sample_val['pred_seasonal_naive'] = s_naive.predict(sample_val)
sample_val['pred_naive'] = naive.predict(sample_val)

fig, ax = plt.subplots(figsize=(15, 5))
ax.plot(sample_val['date'], sample_val['sales'], marker='o', label='Ventas Reales', color='black', linewidth=2.5)
ax.plot(sample_val['date'], sample_val['pred_seasonal_naive'], marker='s', linestyle='--', label='Seasonal Naive (Lag 21)', color='#1f77b4', linewidth=1.8)
ax.plot(sample_val['date'], sample_val['pred_naive'], marker='^', linestyle=':', label='Naive (Lag 16)', color='#ff7f0e', linewidth=1.5)

ax.set_title('Validacion Temporal: Tienda 1 - GROCERY I (2017-07-31 a 2017-08-15)', fontsize=14, weight='bold')
ax.set_xlabel('Fecha')
ax.set_ylabel('Ventas Diarias (Unidades)')
ax.legend(frameon=True)
plt.tight_layout()
plt.show()

## 5. Conclusiones del Benchmark de Referencia

1. **Vara mínima establecida:** El modelo de Machine Learning (LightGBM/XGBoost) debe superar holgadamente el **$RMSLE$** y **$WAPE$** del *Seasonal Naive*.
2. **Causa del error en Baselines:** Los modelos naive no pueden reaccionar a cambios en promociones (`onpromotion`), días festivos específicos ni variaciones del precio del petróleo.
3. **Siguiente paso:** Entrenar el modelo global de Gradient Boosting (`src/retail_platform/models/forecasting/`) utilizando las 54 variables de la Fase 3.